### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="hr_analytics",
    dataset_year="2021",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists",
    download_description="""
mkdir -p local-data-warehouse/hr_analytics && cd local-data-warehouse/hr_analytics && kaggle datasets download arashnic/hr-analytics-job-change-of-data-scientists && unzip hr-analytics-job-change-of-data-scientists.zip && rm hr-analytics-job-change-of-data-scientists.zip && cd ../../
""",
    # References
    academic_reference_bibtex=r"""@misc{arashnic2021hr,
  author       = {Kaggle User Arashnic},
  title        = {HR Analytics: Job Change of Data Scientists},
  year         = {2021},
  howpublished = {\url{https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="arashnic2021hr",
    license="CC0: Public Domain",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- We renamed the target feature and its values to be more descriptive.
- We drop the ID column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LookingForJobChange",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="LookingForJobChange",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "aug_train.csv")

target_feature = "LookingForJobChange"
df = df.rename(columns={"target": target_feature})
df = df.drop(columns=["enrollee_id"])
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})


cat_cols = ['city', 'gender', 'relevent_experience', 'enrolled_university', 'education_level', 'major_discipline', 'experience', 'company_size', 'company_type', 'last_new_job', 'LookingForJobChange']
df[cat_cols] = df[cat_cols].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 19,158
Columns: 13
Use sampling: False (sample size: 19,158)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['training_hours', 'city', 'city_development_index', 'experience', 'company_size', 'last_new_job', 'company_type', 'major_discipline', 'education_level', 'gender']
Rows remaining as candidates after top-10 filter: 208 (of 19,158)

#### Duplicate Report
Total duplicate rows: 49 (0.26% of dataset)
Duplicate rows ignoring target: 74 (0.39% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,LookingForJobChange
0,city_16,0.910,NaN,Has relevent experience,no_enrollment,Graduate,STEM,6,500-999,Pvt Ltd,1,21,No
1,city_103,0.920,Female,Has relevent experience,no_enrollment,Masters,Humanities,>20,100-500,Funded Startup,2,74,No
2,city_104,0.924,NaN,Has relevent experience,no_enrollment,Graduate,STEM,9,10/49,Pvt Ltd,1,94,No
3,city_21,0.624,Male,Has relevent experience,no_enrollment,Masters,STEM,15,10000+,Pvt Ltd,1,75,No
4,city_134,0.698,Male,No relevent experience,no_enrollment,Masters,STEM,12,500-999,NGO,1,157,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,company_type,category,6140.0,32.05,6.0,"Pvt Ltd, Funded Startup, Public Sector, Early Stage Startup, NGO, Other"
1,company_size,category,5938.0,30.99,8.0,"50-99, 100-500, 10000+, 10/49, 1000-4999, <10, 500-999, 5000-9999"
2,gender,category,4508.0,23.53,3.0,"Male, Female, Other"
3,major_discipline,category,2813.0,14.68,6.0,"STEM, Humanities, Other, Business Degree, Arts, No Major"
4,education_level,category,460.0,2.40,5.0,"Graduate, Masters, High School, Phd, Primary School"
5,last_new_job,category,423.0,2.21,6.0,"1, >4, 2, never, 4, 3"
6,enrolled_university,category,386.0,2.01,3.0,"no_enrollment, Full time course, Part time course"
7,experience,category,65.0,0.34,22.0,">20, 5, 4, 3, 6, 2, 7, 10, 9, 8"
8,city,category,0.0,0.00,123.0,"city_103, city_21, city_16, city_114, city_160, city_136, city_67, city_75, city_102, city_104"
9,relevent_experience,category,0.0,0.00,2.0,"Has relevent experience, No relevent experience"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
city_development_index,19158.0,0.828848,0.123362,0.448,0.949
training_hours,19158.0,65.366896,60.058462,1.000,336.000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column              rank                                       
LookingForJobChange 1                          No  14381  75.07
                    2                         Yes   4777  24.93
city                1                    city_103   4355  22.73
                    2                     city_21   2702  14.10
                    3                     city_16   1533   8.00
                    4                    city_114   1336   6.97
                    5                    city_160    845   4.41
company_size        1                        <NA>   5938  30.99
                    2                       50-99   3083  16.09
                    3                     100-500   2571  13.42
                    4                      10000+   2019  10.54
                    5                       10/49   1471   7.68
company_type        1                     Pvt Ltd   9817  51.24
                    2                        <NA>   6140  32.05
                    3              Funded Startup   1001   5.22
                    4               Public Sector    955   4.98
                    5         Early Stage Startup    603   3.15
education_level     1                    Graduate  11598  60.54
                    2                     Masters   4361  22.76
                    3                 High School   2017  10.53
                    4                        <NA>    460   2.40
                    5                         Phd    414   2.16
enrolled_university 1               no_enrollment  13817  72.12
                    2            Full time course   3757  19.61
                    3            Part time course   1198   6.25
                    4                        <NA>    386   2.01
experience          1                         >20   3286  17.15
                    2                           5   1430   7.46
                    3                           4   1403   7.32
                    4                           3   1354   7.07
                    5                           6   1216   6.35
gender              1                        Male  13221  69.01
                    2                        <NA>   4508  23.53
                    3                      Female   1238   6.46
                    4                       Other    191   1.00
last_new_job        1                           1   8040  41.97
                    2                          >4   3290  17.17
                    3                           2   2900  15.14
                    4                       never   2452  12.80
                    5                           4   1029   5.37
major_discipline    1                        STEM  14492  75.64
                    2                        <NA>   2813  14.68
                    3                  Humanities    669   3.49
                    4                       Other    381   1.99
                    5             Business Degree    327   1.71
relevent_experience 1     Has relevent experience  13792  71.99
                    2      No relevent experience   5366  28.01

In [8]:
# Target Distribution
target_df

,count,pct
LookingForJobChange,,
No,14381,75.07
Yes,4777,24.93


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to hr_analytics/019d7368-56c6-75a9-b489-eb6faf376ce5


019d7368-56c6-75a9-b489-eb6faf376ce5
044ad7ad5e65d96f10e6913cd401538e1a1d4c0dd236b1e484e56459fb94425f
